In [0]:
spark.sql("SHOW SCHEMAS IN yelp_dataset").show()

In [0]:
for table in ["yelp_business","yelp_review","yelp_user","yelp_checkin","yelp_tip"]:
    print(table, spark.table(f"yelp_dataset.silver.{table}").count())

In [0]:
business_df = spark.sql("SELECT * FROM yelp_dataset.silver.yelp_business")
display(business_df)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

class GoldETL:
    def __init__(self):
        self.spark = spark
        self.catalog = "yelp_dataset"
        self.silver_schema = "silver"
        self.gold_schema = 'gold'

    def load_silver_data(self, table_name: str) -> DataFrame:
        """Load data from silver layer"""
        table_name_full = f"{self.catalog}.{self.silver_schema}.{table_name}"
        try:
            df = self.spark.table(table_name_full)
            print(f"Loaded {df.count()} records from silver {table_name}")
            return df
        except Exception as e:
            print(f"Failed to load silver data for {table_name}: {e}")
            return None

    def create_business_features(self, business_df: DataFrame, review_df: DataFrame, checkin_df: DataFrame) -> DataFrame:
        """Create comprehensive business features for ML"""
        print("Creating business features")

        review_agg=review_df.groupBy("business_id").agg(
            avg("stars").alias("avg_review_rating"),
            count("review_id").alias("total_reviews"),
            sum("useful").alias("total_useful_votes"),
            sum("funny").alias("total_funny_votes"),
            sum("cool").alias("total_cool_votes"),
            avg("text_length").alias("avg_review_length"),
            stddev("stars").alias("rating_std"),
            min("review_date").alias("first_review_date"),
            max("review_date").alias("last_review_date")
        )

        review_agg=review_agg.withColumn(
            "review_span_months",
            months_between(col("last_review_date"),col("first_review_date"))
        )

        review_agg=review_agg.withColumn(
            "reviews_per_month",
            when(col("review_span_months")>0,
                    col("total_reviews")/col("review_span_months")).otherwise(0)
        )

        checkin_agg=checkin_df.groupBy("business_id").agg(
            sum("checkin_count").alias("total_checkins")
        )

        business_features=business_df.join(
            review_agg,"business_id","left"
        ).join(
            checkin_agg,"business_id","left"
        )

        business_features=business_features.fillna({
            "avg_review_rating":0.0,
            "total_reviews":0,
            "total_useful_votes":0,
            "total_funny_votes":0,
            "total_cool_votes":0,
            "avg_review_length":0.0,
            "rating_std":0.0,
            "reviews_per_month":0.0,
            "total_checkins":0
        })

        business_features=business_features.withColumn(
            "engagement_score",
            col("total_useful_votes")+
            col("total_funny_votes")+
            col("total_cool_votes")
        )

        business_features=business_features.withColumn(
            "popularity_score",
            (
                (col("stars")/5.0)*60+
                (least(col("total_reviews"),lit(1000))/1000.0)*20+
                (least(col("total_checkins"),lit(1000))/1000.0)*20
            ).cast("double")
        )

        # business_features=business_features.withColumn(
        #     "business_tier",
        #     when(col("total_reviews")>=100,"high_volume")
        #     .when(col("total_reviews")>=50,"medium_volume")
        #     .when(col("total_reviews")>=10,"low_volume")
        #     .otherwise("new_business")
        # )

        business_features=business_features.withColumn(
            "rating_consistency",
            when(col("rating_std")<=0.5,"very_consistent")
            .when(col("rating_std")<=1.0,"consistent")
            .when(col("rating_std")<=1.5,"moderate")
            .otherwise("inconsistent")
        )

        # business_features=business_features.withColumn(
        #     "location_key",
        #     concat(col("city"),lit("_"),col("state"))
        # )
        business_features=business_features.withColumn(
        "location_key",concat(col("city_normalized"),lit("_"),col("state_normalized")))

        business_features=business_features.withColumn(
            "processed_timestamp",current_timestamp()
        ).withColumn(
            "data_layer",lit("gold")
        )

        print(f"Business features created: {business_features.count()} records")
        return business_features
    def create_user_features(self, user_df: DataFrame, review_df: DataFrame) -> DataFrame:
        """Create comprehensive user features for CRM analytics"""
        print("Creating user features")

        user_review_agg=review_df.groupBy("user_id").agg(
            count("review_id").alias("user_total_reviews"),
            avg("stars").alias("user_avg_rating"),
            sum("useful").alias("user_total_useful"),
            sum("funny").alias("user_total_funny"),
            sum("cool").alias("user_total_cool"),
            avg("text_length").alias("user_avg_review_length"),
            stddev("stars").alias("user_rating_std"),
            min("review_date").alias("user_first_review"),
            max("review_date").alias("user_last_review"),
            count_distinct("business_id").alias("unique_businesses_reviewed")
        )

        user_review_agg=user_review_agg.withColumn(
            "review_span_days",
            datediff(col("user_last_review"),col("user_first_review"))
        )

        user_review_agg=user_review_agg.withColumn(
            "reviews_per_month",
            when(
                col("review_span_days")>30,
                col("user_total_reviews")/(col("review_span_days")/30.0)
            ).otherwise(0)
        )

        user_review_agg=user_review_agg.withColumn(
            "days_since_last_review",
            datediff(current_date(),col("user_last_review"))
        )

        user_features=user_df.join(user_review_agg,"user_id","left")

        user_features=user_features.fillna({
            "user_total_reviews":0,
            "user_avg_rating":0.0,
            "user_total_useful":0,
            "user_total_funny":0,
            "user_total_cool":0,
            "user_avg_review_length":0.0,
            "user_rating_std":0.0,
            "reviews_per_month":0.0,
            "unique_businesses_reviewed":0,
            "days_since_last_review":9999
        })

        user_features=user_features.withColumn(
            "recency_score",
            when(col("days_since_last_review")<=30,5)
            .when(col("days_since_last_review")<=90,4)
            .when(col("days_since_last_review")<=180,3)
            .when(col("days_since_last_review")<=365,2)
            .otherwise(1)
        )

        user_features=user_features.withColumn(
            "frequency_score",
            when(col("reviews_per_month")>=4,5)
            .when(col("reviews_per_month")>=2,4)
            .when(col("reviews_per_month")>=1,3)
            .when(col("reviews_per_month")>=0.5,2)
            .otherwise(1)
        )

        user_features=user_features.withColumn(
            "monetary_score",
            when(col("engagement_score")>=100,5)
            .when(col("engagement_score")>=50,4)
            .when(col("engagement_score")>=20,3)
            .when(col("engagement_score")>=5,2)
            .otherwise(1)
        )

        user_features=user_features.withColumn(
            "rfm_score",
            concat(
                col("recency_score"),
                col("frequency_score"),
                col("monetary_score")
            )
        )

        user_features=user_features.withColumn(
            "user_segment",
            when(col("rfm_score").isin(
                "555","554","545","544","455","454","445"
            ),"champions")
            .when(col("rfm_score").isin(
                "543","444","435","355","354","345","344","335"
            ),"loyal_customers")
            .when(col("rfm_score").isin(
                "512","511","422","421","412","411","311"
            ),"potential_loyalists")
            .when(col("rfm_score").isin(
                "533","532","531","523","522","521",
                "513","433","432","431","423","413"
            ),"new_customers")
            .when(col("rfm_score").isin(
                "155","154","144","214","215","115","114"
            ),"promising")
            .when(col("rfm_score").isin(
                "254","245","253","252","243","242",
                "235","234","225","224","153","152",
                "145","143","142","135","134","125","124"
            ),"need_attention")
            .when(col("rfm_score").isin(
                "331","321","231","241","251"
            ),"about_to_sleep")
            .when(col("rfm_score").isin(
                "113"
            ),"at_risk")
            .when(col("rfm_score").isin(
                "211","111","112","121","131","141","151"
            ),"cannot_lose_them")
            .otherwise("hibernating")
        )

        user_features=user_features.withColumn(
            "churn_risk",
            when(col("days_since_last_review")>365,"high")
            .when(col("days_since_last_review")>180,"medium")
            .when(col("days_since_last_review")>90,"low")
            .otherwise("active")
        )

        user_features=user_features.withColumn(
            "user_value_score",
            (
                col("user_total_reviews")*0.3+
                col("engagement_score")*0.4+
                col("unique_businesses_reviewed")*0.2+
                col("fans")*0.1
            )
        )

        user_features=user_features.withColumn(
            "processed_timestamp",current_timestamp()
        ).withColumn(
            "data_layer",lit("gold")
        )

        print(f"User features created: {user_features.count()} records")
        return user_features
    def create_review_features(self, review_df: DataFrame, business_df: DataFrame, user_df: DataFrame) -> DataFrame:
        """Create review features for sentiment analysis and NLP"""
        print("Creating review features")

        review_features=review_df.join(
            business_df.select(
                "business_id",
                "name",
                "city",
                "state",
                "categories_array",
                col("stars").alias("business_stars")
            ),
            "business_id",
            "left"
        ).join(
            user_df.select(
                "user_id",
                "user_tier",
                "average_stars",
                "user_tenure_days"
            ),
            "user_id",
            "left"
        )

        review_features=review_features.withColumn(
            "sentence_count",
            size(split(col("text"),r"[.!?]+"))
        )

        review_features=review_features.withColumn(
            "exclamation_count",
            size(split(col("text"),"!"))-1
        )

        review_features=review_features.withColumn(
            "question_count",
            size(split(col("text"),r"\?"))-1
        )

        review_features=review_features.withColumn(
            "caps_ratio",
            when(
                length(col("text"))>0,
                length(regexp_replace(col("text"),"[^A-Z]",""))/
                length(col("text"))
            ).otherwise(0)
        )

        review_features=review_features.withColumn(
            "rating_deviation",
            col("stars")-col("business_stars")
        )

        review_features=review_features.withColumn(
            "user_rating_deviation",
            col("stars")-col("average_stars")
        )

        review_features=review_features.withColumn(
            "review_year",
            year(col("review_date"))
        )

        review_features=review_features.withColumn(
            "review_month",
            month(col("review_date"))
        )

        review_features=review_features.withColumn(
            "review_day_of_week",
            dayofweek(col("review_date"))
        )

        review_features=review_features.withColumn(
            "review_hour",
            hour(col("review_date"))
        )

        review_features=review_features.withColumn(
            "season",
            when(col("review_month").isin(12,1,2),"winter")
            .when(col("review_month").isin(3,4,5),"spring")
            .when(col("review_month").isin(6,7,8),"summer")
            .otherwise("fall")
        )

        review_features=review_features.withColumn(
            "helpfulness_score",
            col("useful")+col("funny")+col("cool")
        )

        review_features=review_features.withColumn(
            "processed_timestamp",current_timestamp()
        ).withColumn(
            "data_layer",lit("gold")
        )

        print(f"Review features created: {review_features.count()} records")
        return review_features
    def create_time_series_features(self, review_df: DataFrame, business_df: DataFrame) -> DataFrame:
        """Create time series features for forecasting"""
        print("Creating time series features")

        monthly_metrics=review_df.join(
            business_df.select(
                "business_id",
                "city",
                "state",
                "categories_array"
            ),
            "business_id",
            "inner"
        ).withColumn(
            "year_month",
            date_format(col("review_date"),"yyyy-MM")
        ).groupBy(
            "year_month",
            "city",
            "state"
        ).agg(
            count("review_id").alias("monthly_reviews"),
            avg("stars").alias("monthly_avg_rating"),
            count_distinct("business_id").alias("active_businesses"),
            count_distinct("user_id").alias("active_users"),
            sum("useful").alias("monthly_useful_votes"),
            avg("text_length").alias("monthly_avg_text_length")
        )

        monthly_metrics=monthly_metrics.withColumn(
            "year",
            year(to_date(col("year_month"),"yyyy-MM"))
        )

        monthly_metrics=monthly_metrics.withColumn(
            "month",
            month(to_date(col("year_month"),"yyyy-MM"))
        )

        window_spec=Window.partitionBy(
            "city","state"
        ).orderBy("year_month")

        monthly_metrics=monthly_metrics.withColumn(
            "prev_month_reviews",
            lag("monthly_reviews",1).over(window_spec)
        )

        monthly_metrics=monthly_metrics.withColumn(
            "review_growth_rate",
            when(
                col("prev_month_reviews")>0,
                (col("monthly_reviews")-col("prev_month_reviews"))/
                col("prev_month_reviews")
            ).otherwise(0)
        )

        monthly_metrics=monthly_metrics.withColumn(
            "reviews_3month_avg",
            avg("monthly_reviews").over(
                window_spec.rowsBetween(-2,0)
            )
        )

        monthly_metrics=monthly_metrics.withColumn(
            "rating_3month_avg",
            avg("monthly_avg_rating").over(
                window_spec.rowsBetween(-2,0)
            )
        )

        monthly_metrics=monthly_metrics.withColumn(
            "processed_timestamp",current_timestamp()
        ).withColumn(
            "data_layer",lit("gold")
        )

        print(f"Time series features created: {monthly_metrics.count()} records")
        return monthly_metrics
    def create_cohort_analysis(self, user_df: DataFrame, review_df: DataFrame) -> DataFrame:
        """Create cohort analysis for user retention"""
        print("Creating cohort analysis")
        user_cohorts=review_df.groupBy("user_id").agg(
            min("review_date").alias("first_review_date")
        ).withColumn(
            "cohort_month",
            date_format(col("first_review_date"),"yyyy-MM")
        )

        cohort_data=review_df.join(
            user_cohorts,
            "user_id"
        ).withColumn(
            "review_month",
            date_format(col("review_date"),"yyyy-MM")
        )

        cohort_data=cohort_data.withColumn(
            "months_since_cohort",
            months_between(
                to_date(col("review_month"),"yyyy-MM"),
                to_date(col("cohort_month"),"yyyy-MM")
            ).cast("int")
        )

        cohort_retention=cohort_data.groupBy(
            "cohort_month",
            "months_since_cohort"
        ).agg(
            count_distinct("user_id").alias("active_users")
        )

        cohort_sizes=user_cohorts.groupBy(
            "cohort_month"
        ).agg(
            count("user_id").alias("cohort_size")
        )

        cohort_analysis=cohort_retention.join(
            cohort_sizes,
            "cohort_month"
        ).withColumn(
            "retention_rate",
            col("active_users")/col("cohort_size")
        )

        cohort_analysis=cohort_analysis.withColumn(
            "processed_timestamp",current_timestamp()
        ).withColumn(
            "data_layer",lit("gold")
        )

        print(f"Cohort analysis created: {cohort_analysis.count()} records")
        return cohort_analysis
    def save_to_gold(self, df: DataFrame, table_name: str):
        """Save DataFrame to gold layer"""
        full_table_name=f"{self.catalog}.{self.gold_schema}.{table_name}"
        df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(full_table_name)
        print(f"Saved {full_table_name}")
    def run(self) -> bool:
        """Run complete gold layer ETL"""
        print("Starting Gold Layer ETL")
        business_df=self.load_silver_data("yelp_business")
        review_df=self.load_silver_data("yelp_review")
        user_df=self.load_silver_data("yelp_user")
        checkin_df=self.load_silver_data("yelp_checkin")
        tip_df=self.load_silver_data("yelp_tip")
        if business_df is None or review_df is None or user_df is None:
            print("Required silver layer data not available")
            return False
        business_features=self.create_business_features(
            business_df,
            review_df,
            checkin_df
        )
        self.save_to_gold(
            business_features,
            "business_features"
        )
        user_features=self.create_user_features(
            user_df,
            review_df
        )
        self.save_to_gold(
            user_features,
            "user_features"
        )
        review_features=self.create_review_features(
            review_df,
            business_df,
            user_df
        )
        self.save_to_gold(
            review_features,
            "review_features"
        )
        time_series_features=self.create_time_series_features(
            review_df,
            business_df
        )
        self.save_to_gold(
            time_series_features,
            "time_series_features"
        )
        cohort_analysis=self.create_cohort_analysis(
            user_df,
            review_df
        )
        self.save_to_gold(
            cohort_analysis,
            "cohort_analysis"
        )
        print("Gold Layer ETL completed successfully")
        return True

In [0]:
gold_etl=GoldETL()
gold_etl.run()

In [0]:
business_df = spark.table("yelp_dataset.gold.business_features")
user_df = spark.table("yelp_dataset.gold.user_features")
review_df = spark.table("yelp_dataset.gold.review_features")
time_series_df = spark.table("yelp_dataset.gold.time_series_features")
cohort_df = spark.table("yelp_dataset.gold.cohort_analysis")

In [0]:
display(business_df.limit(10))

In [0]:
display(user_df.limit(10))

In [0]:
review_df.printSchema()

In [0]:
display(time_series_df.limit(10))

In [0]:
display(cohort_df.limit(10))

In [0]:
cohort_df.select(
    "cohort_month",
    "months_since_cohort",
    "active_users",
    "cohort_size",
    "retention_rate"
).orderBy(
    "cohort_month",
    "months_since_cohort"
).show(30, truncate=False)

In [0]:
cohort_df.select("cohort_month",
    "review_month",
    "months_since_cohort",
    "cohort_size",
    "active_users",
    "retention_rate").show(20,truncate=False)

In [0]:
time_series_df.select(
    "year_month",
    "city",
    "monthly_reviews",
    "monthly_avg_rating",
    "active_businesses",
    "active_users",
    "review_growth_rate"
).show(20, truncate=False)

In [0]:
cohort_df.select(
    "cohort_month",
    "months_since_cohort",
    "active_users",
    "cohort_size",
    "retention_rate"
).orderBy(
    "cohort_month",
    "months_since_cohort"
).show(30, truncate=False)

In [0]:
cohort_df.selectExpr(
    "min(retention_rate) as min_retention",
    "max(retention_rate) as max_retention",
    "avg(retention_rate) as avg_retention"
).show()